## Set up Libraries

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


## Relevance (Accuracy) function

### Generate **Sigmoid** function

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

### Generate **relevance** (y) function of single embedding

In [ ]:
def y(project_emb, member_emb):
    # Eq. (20): y(v,u) = σ(e_v^T e_u)
    dot_prod = np.dot(project_emb, member_emb)
    return sigmoid(dot_prod / 16.0)

### Generate **s_relevance** function (relevance of project v and subset A(k))

In [ ]:
def s_relevance(team_indices, score_dict):
    if not team_indices:
        return 0.0
    return sum(score_dict.get(i, 0.0) for i in team_indices) / len(team_indices)


## Generate candidate set (pool) of **A**

In [ ]:
def build_candidate_set_A_optimized(target_task_index, emb_task, emb_dev, k, b, pinned=None):
    target_task_emb = emb_task[target_task_index]

    # VECTORIZATION: Calculate all 33,621 dot products instantly
    # emb_dev has shape (33621, 256), target_task_emb has shape (256,)
    # The result is an array of 33,621 raw dot products.
    all_dot_products = np.dot(emb_dev, target_task_emb)

    # Apply the scaling and sigmoid to the entire array at once
    all_scores = sigmoid(all_dot_products / 16.0)

    # Create the score_dict using a fast dictionary comprehension
    score_dict = {i: score for i, score in enumerate(all_scores)}

    # Sort and pick the top b * k
    sorted_items = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)
    top_n = int(b * k)
    top = sorted_items[:top_n]

    A = [m_index for (m_index, _) in top]
    # Even if a pinned member scored below the b*k threshold, they MUST be
    # in A so greedy_beam_search can place them and compute diversity correctly.

    if pinned:
      pinned_set= set(pinned)
      existing = set(A)
      missing= pinned_set - existing
      if missing:
        A= list(missing)+A

    return A, score_dict

## **Diversity** function

### Generate diversity function of single embedding

In [ ]:
def sim(u_emb1, u_emb2):
    # Divide by 16 to scale the number down before sigmoid
    dot_prod = np.dot(u_emb1, u_emb2)
    return sigmoid(dot_prod / 16.0)

### Generate diversity function (diversity of all members on team)

In [ ]:
def g_diversity(team_members_indices, emb_dev):
    k = len(team_members_indices) # k is the team size

    if k <= 1:
        return 0.0

    total = 0.0
    count = 0

    for i in range(k):
        for j in range(i):
            u1 = emb_dev[team_members_indices[i]]
            u2 = emb_dev[team_members_indices[j]]

            total += sim(u1, u2)
            count += 1

    avg_sim = total / count

    diversity=1.0-avg_sim
    return diversity

## **Define the Compatibility Function:**

In [ ]:
def compatability_score( team_members_indices,score_dict, emb_dev, alpha):
    relevance = s_relevance( team_members_indices,score_dict)
    diversity = g_diversity(team_members_indices, emb_dev)

    return alpha * relevance + (1 - alpha) * diversity
#now what we make until now is we compute the score of any team but we not get the best team

##**Bounded greedy selection algorithm**

In [ ]:
def greedy_beam_search(target_task_emb, candidate_A, emb_dev, team_size, K, alpha, score_dict, pinned=None):
    pinned_members = list(pinned) if pinned else []

    if len(pinned_members)>team_size:
      #to check that pinned users not more than team size
      raise ValueError (f"Pinned members ({len(pinned)}) exceed team size ({team_size})!")

    remaining= team_size-len(pinned_members)

    #case if pinnes = team size
    if remaining==0:
      return [pinned_members]

    #start with existimg team not empty list
    initial_score= compatability_score(pinned_members, score_dict, emb_dev, alpha) if pinned_members else 0.0
    beam = [(pinned_members, initial_score)]

    #Exclude pinned members from the candidate pool to avoid re-adding them
    new_A = [u for u in candidate_A if u not in set(pinned_members)]

    #fill only the remaining slots
    for step in range(remaining):
        best_by_team = {}  # team_key -> (team_list, score)

        for team, _ in beam:
            for u_idx in candidate_A:
                if u_idx in team:
                    continue

                new_team = team + [u_idx]

                relevance = s_relevance(new_team, score_dict)
                diversity = g_diversity(new_team, emb_dev)
                f_score = alpha * relevance + (1 - alpha) * diversity

                # Canonical key (order-independent)
                team_key = tuple(sorted(new_team))

                # Keep only the best score for the same team set
                if team_key not in best_by_team or f_score > best_by_team[team_key][1]:
                    best_by_team[team_key] = (list(team_key), f_score)

        candidates_for_next_step = list(best_by_team.values())
        candidates_for_next_step.sort(key=lambda x: x[1], reverse=True)
        beam = candidates_for_next_step[:K]

    return [team for team, score in beam]

In [ ]:
def resolve_pinned_members(pinned_member_ids, member_id2idx):
    """
    Converts a list of real member IDs submitted by the user
    into internal embedding indices.
    Warns and skips any ID not found (cold-start or invalid users).
    """
    pinned_indices = []
    skipped = []
    for mid in pinned_member_ids:
        if mid in member_id2idx:
            pinned_indices.append(member_id2idx[mid])
        else:
            skipped.append(mid)

    if skipped:
        print(f"⚠️ Skipped {len(skipped)} unrecognized member IDs: {skipped}")

    return pinned_indices

## Load **Two-Hop** embeddings dataset

In [ ]:
members_emb=np.load("/content/drive/MyDrive/Datasets/team_formation_input/dynamic_version/twohop_train_member_embeddings.npy")
projects_emb=np.load("/content/drive/MyDrive/Datasets/team_formation_input/dynamic_version/twohop_train_project_embeddings.npy")

In [ ]:
member_ids_df  = pd.read_csv("/content/drive/MyDrive/Datasets/team_formation_input/dynamic_version/twohop_train_member_ids.csv")
project_ids_df = pd.read_csv("/content/drive/MyDrive/Datasets/team_formation_input/dynamic_version/twohop_train_project_ids.csv")
member_ids_list  = member_ids_df["member_id"].astype(int).tolist()
project_ids_list = project_ids_df["project_id"].astype(int).tolist()

#mapping id of member to its index
member_id2idx  = {mid: i for i, mid in enumerate(member_ids_list)}
project_id2idx = {pid: i for i, pid in enumerate(project_ids_list)}


### **Exploring Data**

In [ ]:
num_members  = len(member_ids_list)
num_projects = len(project_ids_list)

print("Members:", num_members, "Projects:", num_projects)

Members: 33621 Projects: 18487


In [ ]:
print("projects_emb:", projects_emb.shape)
print("members_emb:", members_emb.shape)
print("project row:", projects_emb[0].shape)
print("member row:", members_emb[0].shape)

projects_emb: (18487, 256)
members_emb: (33621, 256)
project row: (256,)
member row: (256,)


In [ ]:
# there is a problem in the embedings files formatiing need to be fix just small thing when saved them in the files
members_emb  = np.squeeze(members_emb)
projects_emb = np.squeeze(projects_emb)

In [ ]:
train_edges=pd.read_csv('/content/drive/MyDrive/Datasets/team_formation_input/train_edges.csv')

In [ ]:
team_members=pd.read_csv("/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/Cleaned_Dataset/team_members_without_cold.csv")

In [ ]:
team_members = team_members.copy()
train_edges  = train_edges.copy()
team_members["project_id"] = team_members["project_id"].astype(int)
team_members["member_id"]  = team_members["member_id"].astype(int)
train_edges["project_id"]  = train_edges["project_id"].astype(int)
train_edges["member_id"]   = train_edges["member_id"].astype(int)

In [ ]:
# 1) for each project, compare member sets in team_members vs train_edges
tm_sets = (team_members.groupby("project_id")["member_id"]
           .apply(lambda s: set(s.unique())))
tr_sets = (train_edges.groupby("project_id")["member_id"]
           .apply(lambda s: set(s.unique())))

In [ ]:
# align projects that exist in team_members
tr_sets_aligned = tr_sets.reindex(tm_sets.index).apply(lambda x: x if isinstance(x, set) else set())

In [ ]:
# keep projects where team_members' set is subset of train_edges' set
keep_projects = tm_sets.index[tm_sets <= tr_sets_aligned]

In [ ]:
# 2) filter team_members to only these projects
test_team_members = team_members[team_members["project_id"].isin(keep_projects)].copy()

In [ ]:
print("Projects kept:", test_team_members["project_id"].nunique())
print("Rows kept:", len(test_team_members))

Projects kept: 8407
Rows kept: 29439


## **Build the ground-truth team for each project**

In [ ]:
test_team_members['project_id']= test_team_members['project_id'].astype(int)
test_team_members['member_id']= test_team_members['member_id'].astype(int)

In [ ]:
test_team_members.head(10)

,project_id,member_id,member_description
4,1,16897,NaN
5,1,2606,NaN
6,1,15799,NaN
7,1,29405,NaN
8,1,6147,NaN
18,4,27407,NaN
19,4,30874,NaN
20,4,37050,NaN
24,6,32857,NaN
25,6,33020,NaN


In [ ]:
# ground truth
ground_truth = test_team_members.groupby("project_id")['member_id'].apply(set).to_dict()

In [ ]:
print(ground_truth[10])

{996, 33352, 26356, 26902, 26903}


In [ ]:
test_team_members[test_team_members['project_id']==10]

,project_id,member_id,member_description
38,10,26902,NaN
39,10,33352,NaN
40,10,26903,NaN
41,10,26356,NaN
42,10,996,NaN


In [ ]:
print(list(project_id2idx.items())[:5])

[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4)]


## Prepare Hyperparameters

In [ ]:
teams=pd.read_csv("/content/drive/MyDrive/Datasets/2020-2025_Devpost_Dataset/Cleaned_Dataset/teams_without_cold.csv")

In [ ]:
k=4
b = 12
alpha = 0.8
K=12 #number of teams

## Prepare Target Project Id

In [ ]:
def select_top_diverse_teams(candidate_teams, num_to_select=3):

    if not candidate_teams:
        return []

    # 1. Start with the absolute best team (first in the beam)
    selected_teams = [candidate_teams[0]]
    remaining_teams = candidate_teams[1:]

    while len(selected_teams) < num_to_select and remaining_teams:
        best_next_team = None
        max_min_distance = -1.0

        for candidate in remaining_teams:
            # Calculate distance to already selected teams
            # We want the candidate that is most different from the ones we already have
            distances = []
            set_c = set(candidate)

            for selected in selected_teams:
                set_s = set(selected)
                intersection = len(set_c & set_s)
                union_size = len(set_c) + len(set_s)
                # Hamming distance: 1 - (overlap / total_elements)
                dist = 1 - (intersection / union_size)
                distances.append(dist)

            # Diversity Score: How unique is this team compared to our current selection?
            # We use 'min' to ensure it's different from EVERY selected team (Max-Min)
            min_dist = min(distances)

            if min_dist > max_min_distance:
                max_min_distance = min_dist
                best_next_team = candidate

        if best_next_team:
            selected_teams.append(best_next_team)
            remaining_teams.remove(best_next_team)
        else:
            break

    return selected_teams

In [ ]:

##should come from user howa w el pinned ids
target_project_id = int(test_team_members["project_id"].astype(str).iloc[20])
pinned_ids= [33352, 12045]
print("Project Id:", target_project_id)

project_idx = project_id2idx[target_project_id]
pinned_idx= resolve_pinned_members(pinned_ids, member_id2idx)
print("Project index:", project_idx)


Project Id: 10
Project index: 10


In [ ]:
#build candidate pool with pinned members
A, score_dict=build_candidate_set_A_optimized(project_idx, projects_emb, members_emb,k,b, pinned= pinned_idx)


In [ ]:
#topK_teams_indices is indices of members of generated teams
topK_teams_indices = greedy_beam_search(
    projects_emb[project_idx],  # Target Task Embedding
    A,                          # The Bounded Set (Indices)
    members_emb,                # Full embeddings (for Diversity calc)
    k,                          # Team Size
    K,                          # Beam Width (How many teams to find)
    alpha,                      # Balance factor
    score_dict,                  # Pre-calculated scores
    pinned= pinned_idx
)

In [ ]:
top3_diverse_indices = select_top_diverse_teams(topK_teams_indices, num_to_select=3)

# 3. Map to actual IDs for evaluation
top3_actual_ids = [[member_ids_list[idx] for idx in team] for team in top3_diverse_indices]
top3_actual_ids

[[996, 12045, 26902, 33352],
 [12045, 26356, 26903, 33352],
 [12045, 26356, 26902, 33352]]

In [ ]:
topK_teams_indices

[[884, 10559, 23499, 29147],
 [10559, 23006, 23499, 29147],
 [884, 10559, 23500, 29147],
 [10559, 23499, 23500, 29147],
 [10559, 23006, 23500, 29147],
 [884, 10559, 23006, 29147],
 [10559, 19788, 23499, 29147],
 [10559, 19788, 23006, 29147],
 [10559, 19788, 23500, 29147],
 [795, 10559, 23499, 29147],
 [5540, 10559, 23499, 29147],
 [10559, 23499, 29147, 29667]]

In [ ]:
print("Number of generated teams:", len(topK_teams_indices))

Number of generated teams: 12


In [ ]:
#map indices of members of generated teams to actual IDs
topK_actual_ids = []
for team in topK_teams_indices:
    actual_ids = [member_ids_list[idx] for idx in team]
    topK_actual_ids.append(actual_ids)


In [ ]:
topK_actual_ids

[[996, 12045, 26902, 33352],
 [12045, 26356, 26902, 33352],
 [996, 12045, 26903, 33352],
 [12045, 26902, 26903, 33352],
 [12045, 26356, 26903, 33352],
 [996, 12045, 26356, 33352],
 [12045, 22648, 26902, 33352],
 [12045, 22648, 26356, 33352],
 [12045, 22648, 26903, 33352],
 [893, 12045, 26902, 33352],
 [6269, 12045, 26902, 33352],
 [12045, 26902, 33352, 33943]]

In [ ]:
test_team_members[test_team_members['member_id']==33352]

,project_id,member_id,member_description
39,10,33352,NaN
5045,1405,33352,NaN


## **Evaluation**

In [ ]:
def precision_one_team(pred_team_ids, true_team_ids):
    pred_set = set(pred_team_ids)
    true_set = set(true_team_ids)

    if len(pred_set) == 0:
        return 0.0

    return len(pred_set & true_set) / len(pred_set)


In [ ]:
def mean_precision_at_K(topK_teams, true_team_ids):
    if len(topK_teams) == 0:
        return 0.0

    precisions = [
        precision_one_team(team, true_team_ids)
        for team in topK_teams
    ]

    return np.mean(precisions)


In [ ]:
true_set = ground_truth[target_project_id]

mp = mean_precision_at_K(topK_actual_ids, true_set)

print("MP@K for this project:", mp)

MP@K for this project: 0.625


In [ ]:
def recall_one_team(pred_team_ids, true_team_ids):
    """Eq. (MR@K) calculation for one instance."""
    pred_set = set(pred_team_ids)
    true_set = set(true_team_ids)

    if len(true_set) == 0:
        return 0.0

    # Correct recommendations / acual size of ground
    return len(pred_set & true_set) / len(true_set)

In [ ]:
def mild_score(topK_teams):
    """Calculates Mean Inter-List Diversity using Hamming Distance[cite: 407, 408]."""
    if len(topK_teams) < 2:
        return 0.0

    h_distances = []
    num_teams = len(topK_teams)

    for i in range(num_teams):
        for j in range(i + 1, num_teams):
            set_i = set(topK_teams[i])
            set_j = set(topK_teams[j])

            # Hamming Distance Eq. (25): 1 - (intersection / sum of lengths) [cite: 408]
            intersection = len(set_i & set_j)
            union_size = len(set_i) + len(set_j)

            h_ij = 1 - (intersection / union_size) if union_size > 0 else 0
            h_distances.append(h_ij)

    return np.mean(h_distances) if h_distances else 0.0

### Calculating overall MP, MR, MILD for all teams

In [ ]:
import matplotlib.pyplot as plt

def evaluate_model_for_K_values(K_list, test_projects_subset, projects_emb, members_emb, member_ids_list, k_size, b, alpha):
    """
    Runs the full evaluation pipeline for a list of K values and returns the scores for charting.
    """
    # Dictionaries to hold our final plotted data
    plot_data = {
        "MP": [],
        "MR": [],
        "MILD": []
    }

    for current_K in K_list:
        print(f"\nEvaluating for K = {current_K}...")

        results_MP = []
        results_MR = []
        results_MILD = []

        for pid, true_set in test_projects_subset:
            if not true_set or pid not in project_id2idx:
                continue

            project_idx = project_id2idx[pid]

            # 1. Get Candidate Pool (Using the blazing-fast vectorized version)
            A, score_dict = build_candidate_set_A_optimized(project_idx, projects_emb, members_emb, k_size, b)

            # 2. Run Beam Search for the specific current_K
            topK_team_indices = greedy_beam_search(
                target_task_emb=projects_emb[project_idx],
                candidate_A=A,
                emb_dev=members_emb,
                team_size=k_size,
                K=current_K,
                alpha=alpha,
                score_dict=score_dict
            )

            # 3. Map indices to actual IDs
            topK_actual_ids = [[member_ids_list[idx] for idx in team] for team in topK_team_indices]

            # 4. Calculate metrics for this project
            results_MP.append(mean_precision_at_K(topK_actual_ids, true_set))

         # Cumulative Mean Recall (Union of all recommended members)
            union_predicted_devs = set(dev for team in topK_actual_ids for dev in team)
            if len(true_set) > 0:
                cumulative_recall = len(union_predicted_devs & true_set) / len(true_set)
            else:
                cumulative_recall = 0.0
            results_MR.append(cumulative_recall)

            results_MILD.append(mild_score(topK_actual_ids))

        # Calculate the grand average for this specific K value
        avg_mp = np.mean(results_MP)
        avg_mr = np.mean(results_MR)
        avg_mild = np.mean(results_MILD)

        # Store for charting
        plot_data["MP"].append(avg_mp)
        plot_data["MR"].append(avg_mr)
        plot_data["MILD"].append(avg_mild)

        print(f"Done! -> MP@{current_K}: {avg_mp:.4f} | MR@{current_K}: {avg_mr:.4f} | MILD@{current_K}: {avg_mild:.4f}")

    return plot_data

# --- RUN THE AUTOMATED EVALUATION ---
K_values_to_test = [1,2,3,4,5,6,7,8,9,10,11,12]

test_projects_subset = list(ground_truth.items())[:8400]

# Note: Using your already defined variables (k, b, alpha)
final_plot_data = evaluate_model_for_K_values(
    K_list=K_values_to_test,
    test_projects_subset=test_projects_subset,
    projects_emb=projects_emb,
    members_emb=members_emb,
    member_ids_list=member_ids_list,
    k_size=k,
    b=b,
    alpha=alpha
)


Evaluating for K = 1...
Done! -> MP@1: 0.7459 | MR@1: 0.8632 | MILD@1: 0.0000

Evaluating for K = 2...
Done! -> MP@2: 0.7165 | MR@2: 0.9177 | MILD@2: 0.6306

Evaluating for K = 3...
Done! -> MP@3: 0.7045 | MR@3: 0.9421 | MILD@3: 0.6400

Evaluating for K = 4...
Done! -> MP@4: 0.6951 | MR@4: 0.9548 | MILD@4: 0.6474

Evaluating for K = 5...
Done! -> MP@5: 0.6885 | MR@5: 0.9631 | MILD@5: 0.6533

Evaluating for K = 6...
Done! -> MP@6: 0.6832 | MR@6: 0.9688 | MILD@6: 0.6579

Evaluating for K = 7...
Done! -> MP@7: 0.6791 | MR@7: 0.9733 | MILD@7: 0.6620

Evaluating for K = 8...
Done! -> MP@8: 0.6754 | MR@8: 0.9766 | MILD@8: 0.6655

Evaluating for K = 9...
Done! -> MP@9: 0.6722 | MR@9: 0.9787 | MILD@9: 0.6684

Evaluating for K = 10...
Done! -> MP@10: 0.6694 | MR@10: 0.9812 | MILD@10: 0.6711

Evaluating for K = 11...
Done! -> MP@11: 0.6669 | MR@11: 0.9828 | MILD@11: 0.6737

Evaluating for K = 12...
Done! -> MP@12: 0.6648 | MR@12: 0.9842 | MILD@12: 0.6758


### Visualization

In [ ]:
def plot_csdrec_metrics(K_values, plot_data):
    """
    Generates a 1x3 grid of line charts for MP, MR, and MILD.
    """
    # Create a figure with 3 subplots side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Common styling configurations
    marker_style = 's-' # Square markers with solid line, exactly like the paper
    color = '#1f77b4'   # A nice deep blue
    linewidth = 2
    markersize = 8

    # --- Chart 1: Mean Precision (MP) ---
    axes[0].plot(K_values, plot_data["MP"], marker_style, color=color, linewidth=linewidth, markersize=markersize, label="CsdRec")
    axes[0].set_title('Mean Precision (MP) vs. K', fontsize=14, pad=10)
    axes[0].set_xlabel('K (Number of Recommended Teams)', fontsize=12)
    axes[0].set_ylabel('MP', fontsize=12)
    axes[0].set_xticks(K_values)
    axes[0].grid(True, linestyle='--', alpha=0.7)
    axes[0].legend(loc="best")

    # --- Chart 2: Mean Recall (MR) ---
    axes[1].plot(K_values, plot_data["MR"], marker_style, color=color, linewidth=linewidth, markersize=markersize, label="CsdRec")
    axes[1].set_title('Mean Recall (MR) vs. K', fontsize=14, pad=10)
    axes[1].set_xlabel('K (Number of Recommended Teams)', fontsize=12)
    axes[1].set_ylabel('MR', fontsize=12)
    axes[1].set_xticks(K_values)
    axes[1].grid(True, linestyle='--', alpha=0.7)
    axes[1].legend(loc="best")

    # --- Chart 3: Mean Inter-List Diversity (MILD) ---
    axes[2].plot(K_values, plot_data["MILD"], marker_style, color=color, linewidth=linewidth, markersize=markersize, label="CsdRec")
    axes[2].set_title('Mean Diversity (MILD) vs. K', fontsize=14, pad=10)
    axes[2].set_xlabel('K (Number of Recommended Teams)', fontsize=12)
    axes[2].set_ylabel('MILD', fontsize=12)
    axes[2].set_xticks(K_values)
    axes[2].grid(True, linestyle='--', alpha=0.7)
    axes[2].legend(loc="best")

    # Adjust layout so everything fits perfectly
    plt.tight_layout()
    plt.show()

# --- GENERATE THE CHARTS ---
plot_csdrec_metrics(K_values_to_test, final_plot_data)